# Backtester Core Engine

**Owner:** James  
**Version:** 2.0 (Feb 13, 2026)  

Core backtesting engine for evaluating sports betting strategies.  
All logic is imported from `src/cuic_quant/backtest/backtester_backend.py`.  
See `docs/reference/strategy-interface.md` for the strategy function contract.

## Output Format

| Column | Type | Description |
|--------|------|-------------|
| timestamp | datetime | When trade happened |
| game | str | "Home vs Away" |
| action | str | 'BUY_HOME' or 'BUY_AWAY' |
| bet_size | float | Dollars bet |
| odds | float | Decimal odds used |
| outcome | str | 'WIN' or 'LOSS' |
| pnl | float | Profit/loss for this trade |
| cumulative_pnl | float | Running total P&L |
| bankroll | float | Current bankroll after trade |

In [1]:
import sys
from pathlib import Path

# Ensure src/ is importable when running from tools/ directory
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "tools":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from cuic_quant.backtest import (
    load_backtest_data,
    backtest,
    always_bet_home,
    validate_backtest_results,
)

## Configuration

In [2]:
DATA_DIR = PROJECT_ROOT / "data"
DUMMY_CSV = DATA_DIR / "dummy_backtest_input.csv"
TEST_CSV = DATA_DIR / "test_games.csv"

## Load Data

In [3]:
data = load_backtest_data("2026-01-01", "2026-01-31", csv_path=DUMMY_CSV)
print(f"Games loaded: {len(data)}")
data.head()

Loaded 25 rows from dummy_backtest_input.csv.
Games loaded: 25


,timestamp,game,home_team,away_team,home_odds,away_odds,home_win
0,2026-01-01,Lakers vs Celtics,Lakers,Celtics,1.95,2.05,1
1,2026-01-02,Warriors vs Heat,Warriors,Heat,1.80,2.20,1
2,2026-01-03,Bucks vs Nets,Bucks,Nets,1.65,2.45,1
3,2026-01-04,Nuggets vs Suns,Nuggets,Suns,2.10,1.85,0
4,2026-01-05,76ers vs Knicks,76ers,Knicks,1.90,2.00,1


## Strategy

Using the built-in `always_bet_home` test strategy.

To use your own strategy, replace the assignment below:

```python
def my_strategy(row, context=None):
    return {"action": "BUY_HOME", "confidence": 0.8, "size": 50.0}

strategy = my_strategy
```

See `docs/reference/strategy-interface.md` for the full contract.

In [4]:
strategy = always_bet_home

## Run Backtest

In [5]:
results = backtest(data, strategy, initial_bankroll=10000.0)
print(f"Total trades: {len(results)}")
results

Total trades: 25


,timestamp,game,action,bet_size,odds,outcome,pnl,cumulative_pnl,bankroll
0,2026-01-01,Lakers vs Celtics,BUY_HOME,100.0,1.95,WIN,95.0,95.0,10095.0
1,2026-01-02,Warriors vs Heat,BUY_HOME,100.0,1.80,WIN,80.0,175.0,10175.0
2,2026-01-03,Bucks vs Nets,BUY_HOME,100.0,1.65,WIN,65.0,240.0,10240.0
3,2026-01-04,Nuggets vs Suns,BUY_HOME,100.0,2.10,LOSS,-100.0,140.0,10140.0
4,2026-01-05,76ers vs Knicks,BUY_HOME,100.0,1.90,WIN,90.0,230.0,10230.0
5,2026-01-06,Celtics vs Heat,BUY_HOME,100.0,1.70,WIN,70.0,300.0,10300.0
6,2026-01-07,Lakers vs Warriors,BUY_HOME,100.0,2.15,LOSS,-100.0,200.0,10200.0
7,2026-01-08,Nets vs Bucks,BUY_HOME,100.0,2.40,LOSS,-100.0,100.0,10100.0
8,2026-01-09,Suns vs 76ers,BUY_HOME,100.0,1.95,WIN,95.0,195.0,10195.0
9,2026-01-10,Knicks vs Nuggets,BUY_HOME,100.0,2.25,LOSS,-100.0,95.0,10095.0


## Summary Statistics

In [6]:
if len(results) > 0:
    wins = (results["outcome"] == "WIN").sum()
    losses = (results["outcome"] == "LOSS").sum()
    win_rate = wins / len(results)
    final_pnl = results["cumulative_pnl"].iloc[-1]
    final_bankroll = results["bankroll"].iloc[-1]

    print(f"Win Rate:        {win_rate:.1%} ({wins}W / {losses}L)")
    print(f"Total P&L:       ${final_pnl:,.2f}")
    print(f"Final Bankroll:  ${final_bankroll:,.2f}")
    print(f"ROI:             {final_pnl / 10000:.1%}")
else:
    print("No trades executed.")

Win Rate:        60.0% (15W / 10L)
Total P&L:       $305.00
Final Bankroll:  $10,305.00
ROI:             3.0%


## Validate Results

Run the full validation suite to check schema, math correctness, and data leakage.

In [7]:
report = validate_backtest_results(results, data)

if report["passed"]:
    print(f"PASSED: {report['checks_passed']}/{report['checks_run']} checks passed")
else:
    print(f"FAILED: {report['checks_passed']}/{report['checks_run']} checks passed")
    for f in report["failures"]:
        print(f"  - {f}")

PASSED: 12/12 checks passed


## Test with Mya's `test_games.csv`

Run the backtester against Mya's 100-row test dataset to validate compatibility.

In [ ]:
if TEST_CSV.exists():
    mya_data = load_backtest_data("2026-01-01", "2026-12-31", csv_path=TEST_CSV)
    mya_results = backtest(mya_data, strategy, initial_bankroll=10000.0)
    mya_report = validate_backtest_results(mya_results, mya_data)

    print(f"Mya's data: {len(mya_results)} trades\n")

    # Summary stats
    if len(mya_results) > 0:
        wins = (mya_results["outcome"] == "WIN").sum()
        losses = (mya_results["outcome"] == "LOSS").sum()
        win_rate = wins / len(mya_results)
        final_pnl = mya_results["cumulative_pnl"].iloc[-1]
        final_bankroll = mya_results["bankroll"].iloc[-1]

        print(f"Win Rate:        {win_rate:.1%} ({wins}W / {losses}L)")
        print(f"Total P&L:       ${final_pnl:,.2f}")
        print(f"Final Bankroll:  ${final_bankroll:,.2f}")
        print(f"ROI:             {final_pnl / 10000:.1%}\n")

    # Validation
    print(f"Validation: {'PASSED' if mya_report['passed'] else 'FAILED'}")
    print(f"Checks: {mya_report['checks_passed']}/{mya_report['checks_run']}")
    if not mya_report["passed"]:
        print("\nFailed checks:")
        for f in mya_report["failures"]:
            print(f"  - {f}")

    # Display results table
    display(mya_results)
else:
    print(f"test_games.csv not found at {TEST_CSV}")